# 01 -- Data quality

**Read this before anything else in the project.** The build order in the design
brief is deliberate: the composite signal does not get built until the
per-analyst data has been shown to support it. This notebook is that gate.

It is a thin wrapper. Every check lives in `src/diagnostics/data_quality.py`
and is unit-tested in `tests/`; the notebook exists to look at the numbers, not
to hold the logic. That way the gate cannot quietly diverge from what the
pipeline actually enforces.

What the checks are looking for, in rough order of how much damage each does if
missed:

| check | the failure it catches |
|---|---|
| vendor `spot_at_action` integrity | vendor back-fills that column with *today's* price, making every implied return near-zero by construction |
| corporate-action integrity | targets stored as-quoted against a back-adjusted price series -- a 4:1 split turns +5% into +320% |
| coverage depth | fewer than four firms: there is no crowd to aggregate |
| `action_date` is a trading date | the date column is the vendor's publication timestamp, not when the analyst acted |
| staleness | firms that stopped updating still appear in a naive consensus |
| vendor restatement rate | the vendor has rewritten history since capture |
| vendor agreement | two vendors disagree on the same event, so the panel inherits one vendor's inclusion rules invisibly |


In [ ]:
import sys, warnings
from datetime import date, timedelta
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src import config
from src.store import pit, writer

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

TICKER = "AAPL"
ASOF = date.today()

# 'strict' is the only mode whose output is safe to publish. On a store built
# from a single snapshot it returns nothing for past as-of dates, which is the
# truthful answer -- see src/store/pit.py. Switch to ASSUME_VENDOR_HISTORY only
# with that assumption in mind.
PIT_MODE = pit.ASSUME_VENDOR_HISTORY

con = writer.connect(config.DB_PATH, read_only=True)
prov = pit.provenance(con, TICKER)
prov


## The gate

Critical failures mean stop. Warnings mean know what you are carrying.

In [ ]:
from src.diagnostics.data_quality import run_quality_checks, render_quality_report

report = run_quality_checks(con, TICKER, ASOF, pit_mode=PIT_MODE)
print(render_quality_report(report))
print("\nGATE PASSED" if report.passed else "\nGATE FAILED -- do not proceed to the composite signal")


## The numbers behind each check

Every check carries its own numbers so a borderline result can be judged rather than accepted.

In [ ]:
rows = []
for c in report.checks:
    for k, v in c.numbers.items():
        rows.append({"check": c.name, "severity": c.severity, "passed": c.passed,
                     "metric": k, "value": v})
pd.DataFrame(rows)


## Coverage over time

How many distinct firms had a live target in the trailing 180-day window, month by month. A panel that thins out is a panel whose later years cannot support the same aggregation as its earlier ones.

In [ ]:
panel = pit.price_targets_asof(con, TICKER, ASOF, pit_mode=PIT_MODE)
history = pit.price_history_asof(con, TICKER, ASOF)

rows = []
for ts in pd.date_range(min(panel["action_date"]), ASOF, freq="ME"):
    t = ts.date()
    window = panel[(panel["action_date"] <= t) &
                   (panel["action_date"] > t - timedelta(days=config.MAX_LEVEL_AGE_DAYS))]
    rows.append({"date": t,
                 "n_records": len(window),
                 "n_firms": window["analyst_firm"].nunique()})
coverage = pd.DataFrame(rows).set_index("date")
coverage.tail(24)


In [ ]:
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(11, 3.5))
    coverage["n_firms"].plot(ax=ax, lw=1.6)
    ax.axhline(4, ls="--", c="crimson", lw=1)
    ax.set_title(f"{TICKER}: distinct firms with a target inside the "
                 f"{config.MAX_LEVEL_AGE_DAYS}-day window")
    ax.set_ylabel("firms")
    ax.text(coverage.index[5], 4.2, "abstention floor (n_eff < 4)", color="crimson", fontsize=8)
    plt.tight_layout()
except ImportError:
    print("matplotlib not installed; the table above carries the same information")


## Staleness

The question is not how old the archive is -- it is how old the panel you would aggregate *today* is. Measured as the age of each covering firm's most recent action.

In [ ]:
from src.model.debias import attach_implied_returns

df = attach_implied_returns(panel, history)
df["age_days"] = [(ASOF - d).days for d in df["action_date"]]

latest = (df[df["usable"] & (df["age_days"] <= 365)]
          .groupby("analyst_firm")["age_days"].min()
          .sort_values())
print(f"{len(latest)} firms with an action in the last year")
print(f"median firm's latest action: {latest.median():.0f} days old")
print(f"share of firms beyond the {config.MAX_LEVEL_AGE_DAYS}-day cutoff: "
      f"{(latest > config.MAX_LEVEL_AGE_DAYS).mean():.0%}")
latest.to_frame("days_since_last_action")


## Vendor disagreement

Aggregators disagree materially with one another on the same day, because they
use different staleness windows and different panel-inclusion rules. Consuming
any of their consensus figures imports that methodology silently -- which is why
this pipeline ingests per-analyst records and never a consensus.

With one source in the store this cell can only report that the comparison is
impossible, which is itself worth seeing.

In [ ]:
sources = sorted(panel["source"].unique())
print("sources in the store:", sources)

if len(sources) >= 2:
    wide = panel.pivot_table(index=["analyst_firm", "action_date"],
                             columns="source", values="price_target", aggfunc="first")
    both = wide.dropna()
    print(f"{len(both)} events covered by more than one vendor")
    if len(both):
        a, b = both.iloc[:, 0], both.iloc[:, 1]
        disagreement = (a / b - 1).abs()
        print(f"disagree on the target in {(disagreement > 0.001).mean():.1%} of cases")
        print(disagreement.describe())
else:
    print("Only one source present -- cross-vendor disagreement cannot be measured.")
    print("A single-source panel inherits that vendor's inclusion rules invisibly,")
    print("and there is no way to detect it from inside the panel.")


## Target granularity

Sell-side targets cluster hard on round numbers. That matters downstream: a
kernel density estimated over them is smoothing a coarse grid, and the apparent
smoothness of the resulting distribution overstates how finely the panel
actually disagrees.

In [ ]:
pts = df.loc[df["price_target_used"].notna(), "price_target_used"]
print(f"{len(pts)} targets, {pts.nunique()} distinct values")
print(f"share that are exact multiples of 5:  {np.isclose(pts % 5, 0).mean():.0%}")
print(f"share that are exact multiples of 10: {np.isclose(pts % 10, 0).mean():.0%}")
pts.describe()
